# Customer Segmentation: Data Understanding

## Objective

Understand the structure, quality, and characteristics of the raw transaction data before performing any cleaning or customer-level feature engineering.

## Questions

- How many transactions are in the dataset?
- What columns are available?
- What are the data types?
- How many unique customers and products are present?
- Are there missing values?
- Are there cancelled transactions?
- What is the overall date range?

In [13]:
import pandas as pd

file_path = "../data/raw/Online_Retail.xlsx"

df = pd.read_excel(file_path)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [14]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.columns.tolist())

Rows: 541909
Columns: 8
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [15]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [16]:
missing = df.isna().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": (missing / len(df) * 100).round(2)
})

missing_summary

,missing_count,missing_percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


In [17]:
customer_id_summary = pd.DataFrame({
    "CustomerID_Status": ["Known", "Missing"],
    "Transaction_Count": [
        df["CustomerID"].notna().sum(),
        df["CustomerID"].isna().sum()
    ]
})

customer_id_summary["Percentage"] = (
    customer_id_summary["Transaction_Count"]
    / len(df)
    * 100
).round(2)

customer_id_summary


,CustomerID_Status,Transaction_Count,Percentage
0,Known,406829,75.07
1,Missing,135080,24.93


In [18]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

customer_id_revenue = (
    df.assign(
        CustomerID_Status=df["CustomerID"].notna()
        .map({True: "Known", False: "Missing"})
    )
    .groupby("CustomerID_Status")
    .agg(
        Transaction_Count=("InvoiceNo", "count"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

customer_id_revenue["Revenue_Percentage"] = (
    customer_id_revenue["Total_Revenue"]
    / customer_id_revenue["Total_Revenue"].sum()
    * 100
).round(2)

customer_id_revenue

,CustomerID_Status,Transaction_Count,Total_Revenue,Revenue_Percentage
0,Known,406829,8300065.814,85.15
1,Missing,135080,1447682.120,14.85


In [19]:
cancellation_summary = pd.DataFrame({
    "Transaction_Status": ["Normal", "Cancelled"],
    "Transaction_Count": [
        (~df["InvoiceNo"].astype(str).str.startswith("C")).sum(),
        df["InvoiceNo"].astype(str).str.startswith("C").sum()
    ]
})

cancellation_summary["Percentage"] = (
    cancellation_summary["Transaction_Count"]
    / len(df)
    * 100
).round(2)

cancellation_summary

,Transaction_Status,Transaction_Count,Percentage
0,Normal,532621,98.29
1,Cancelled,9288,1.71


In [20]:
validity_summary = pd.DataFrame({
    "Check": [
        "Quantity <= 0",
        "UnitPrice <= 0"
    ],
    "Count": [
        (df["Quantity"] <= 0).sum(),
        (df["UnitPrice"] <= 0).sum()
    ]
})

validity_summary["Percentage"] = (
    validity_summary["Count"] / len(df) * 100
).round(2)

validity_summary

,Check,Count,Percentage
0,Quantity <= 0,10624,1.96
1,UnitPrice <= 0,2517,0.46


In [21]:
date_summary = pd.DataFrame({
    "Metric": ["Start Date", "End Date", "Total Days"],
    "Value": [
        df["InvoiceDate"].min(),
        df["InvoiceDate"].max(),
        (df["InvoiceDate"].max() - df["InvoiceDate"].min()).days
    ]
})

date_summary

,Metric,Value
0,Start Date,2010-12-01 08:26:00
1,End Date,2011-12-09 12:50:00
2,Total Days,373


## Initial Data Understanding Summary

The Online Retail dataset contains 541,909 transaction records across 8 columns, covering approximately 373 days from December 1, 2010 to December 9, 2011.

### Key Findings

- 75.07% of transactions have a known CustomerID, while 24.93% have a missing CustomerID.
- Transactions with known CustomerID account for 85.15% of total revenue.
- Transactions with missing CustomerID account for 14.85% of total revenue.
- 1.71% of transactions are cancellations, identified by InvoiceNo values beginning with "C".
- 1.96% of records have Quantity <= 0.
- 0.46% of records have UnitPrice <= 0.
- Missing CustomerID and invalid transaction values require further investigation during the data cleaning phase.

### Initial Business Implication

Customer-level segmentation should primarily use transactions with identifiable CustomerIDs, because customer-level behavior cannot be reliably attributed when CustomerID is missing.

However, missing-CustomerID transactions should not be ignored from overall business analysis because they represent 14.85% of total recorded revenue.

Invalid quantities, zero/negative prices, and cancelled transactions will be handled systematically during Phase 2: Data Cleaning.